# DETOURA — Regional Travel Intelligence Platform

## MODEL 1 — Content-Based Destination Recommender

### Objective

Build a cold-start destination recommendation engine that matches a
traveler's preferences with destinations across West Bengal.

### Core Preference Space

Each traveler and destination is represented using a 4-dimensional vector:

**[Nature, Adventure, Culture, Relaxation]**

### Mathematical Model

The similarity between a user preference vector `u` and destination vector
`vᵢ` is calculated using Cosine Similarity:

$$
Sim(u,v_i) =
\frac{u \cdot v_i}
{\|u\| \times \|v_i\|}
$$

### Pipeline

User Preferences  
→ 4D Preference Vector  
→ Cosine Similarity  
→ Compare Against 515 Destinations  
→ Rank Destinations  
→ Top-K Recommendations

### Model 1 Output

The model produces a ranked list of destinations based purely on
preference similarity.

### Scope

Model 1 intentionally does not consider:

- Crowd Pressure (CPI)
- Underrated Score (AUS)
- Seasonal Suitability (SSI)
- Generative AI / Gemini

These factors are introduced in **Model 2 — DETOURA Intelligence Engine**.

### Validation Criteria

- 515 destinations successfully loaded
- 4-dimensional preference vectors
- Similarity values within `[0, 1]`
- Correct descending ranking
- Reusable recommendation function
- Multiple preference profiles tested

In [171]:
# ============================================
# DETOURA — MODEL 1
# Content-Based Destination Recommender
# ============================================

import os
import ast
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

print("============================================")
print("DETOURA — MODEL 1")
print("Content-Based Destination Recommender")
print("============================================")

DETOURA — MODEL 1
Content-Based Destination Recommender


In [172]:
# ============================================
# CELL 2 — LOCATE DETOURA DATASET
# ============================================

import os

print("Searching Kaggle input directory...\n")

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

Searching Kaggle input directory...

/kaggle/input/datasets/dragold5000/detoura-west-bengal-master-v2/detoura_west_bengal_master-v2.csv


In [173]:
# ============================================
# CELL 3 — LOAD DETOURA DATASET
# ============================================

csv_files = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print("CSV files found:")
for path in csv_files:
    print(" -", path)

if not csv_files:
    raise FileNotFoundError("No CSV file found in /kaggle/input")

# Prefer the DETOURA master dataset
master_files = [
    path for path in csv_files
    if "master-v2" in os.path.basename(path).lower()
]

CSV_PATH = master_files[0] if master_files else csv_files[0]

print("\nUsing:")
print(CSV_PATH)

df = pd.read_csv(CSV_PATH)

print("\nDataset loaded successfully!")
print("Shape:", df.shape)

CSV files found:
 - /kaggle/input/datasets/dragold5000/detoura-west-bengal-master-v2/detoura_west_bengal_master-v2.csv

Using:
/kaggle/input/datasets/dragold5000/detoura-west-bengal-master-v2/detoura_west_bengal_master-v2.csv

Dataset loaded successfully!
Shape: (515, 24)


In [174]:
# ============================================
# CELL 4 — DATASET VALIDATION
# ============================================

EXPECTED_ROWS = 515
EXPECTED_COLUMNS = 24

print("Rows:", len(df))
print("Columns:", len(df.columns))

assert len(df) == EXPECTED_ROWS, (
    f"Expected {EXPECTED_ROWS} rows, found {len(df)}"
)

assert len(df.columns) == EXPECTED_COLUMNS, (
    f"Expected {EXPECTED_COLUMNS} columns, found {len(df.columns)}"
)

print("\n✓ Row count validated")
print("✓ Column count validated")
print("✓ DETOURA master dataset is ready")

Rows: 515
Columns: 24

✓ Row count validated
✓ Column count validated
✓ DETOURA master dataset is ready


In [175]:
# ============================================
# CELL 5 — INSPECT DATASET SCHEMA
# ============================================

print("DETOURA DATASET COLUMNS")
print("=" * 50)

for i, column in enumerate(df.columns, start=1):
    print(f"{i:02d}. {column}")
# Preview the dataset
display(df.head(5))

DETOURA DATASET COLUMNS
01. destination_id
02. name
03. district
04. category
05. latitude
06. longitude
07. elevation_m
08. slope_deg
09. avg_ndvi
10. accessibility_km_to_nh
11. osm_hotel_count
12. osm_attraction_count
13. osm_restaurant_count
14. historical_review_volume_proxy
15. budget_tier
16. estimated_daily_inr
17. nature
18. adventure
19. culture
20. relaxation
21. crowd_pressure_index
22. underrated_score
23. climate_temp_c
24. climate_rain_mm


,destination_id,name,district,category,latitude,longitude,elevation_m,slope_deg,avg_ndvi,accessibility_km_to_nh,...,budget_tier,estimated_daily_inr,nature,adventure,culture,relaxation,crowd_pressure_index,underrated_score,climate_temp_c,climate_rain_mm
0,WB_DAR_001,Darjeeling,Darjeeling,Hill Station,27.0360,88.2627,2042.0,22.4,0.65,1.2,...,2,3500,0.90,0.70,0.8,0.6,0.272,0.522,"[8.5, 10.2, 14.1, 16.8, 18.0, 19.1, 19.5, 19.4...","[12, 15, 30, 75, 180, 480, 700, 620, 390, 80, ..."
1,WB_DAR_002,Lepchajagat,Darjeeling,Hill Station,27.0135,88.1965,2123.0,18.2,0.82,8.5,...,1,1800,0.95,0.50,0.3,0.9,0.025,0.255,"[8.5, 10.2, 14.1, 16.8, 18.0, 19.1, 19.5, 19.4...","[12, 15, 30, 75, 180, 480, 700, 620, 390, 80, ..."
2,WB_DAR_003,Takdah,Darjeeling,Hill Station,27.0384,88.3639,1600.0,19.5,0.78,12.4,...,1,1600,0.95,0.60,0.4,0.9,0.017,0.247,"[8.5, 10.2, 14.1, 16.8, 18.0, 19.1, 19.5, 19.4...","[12, 15, 30, 75, 180, 480, 700, 620, 390, 80, ..."
3,WB_DAR_004,Sandakphu,Darjeeling,Hill Station,27.1051,88.0016,3636.0,28.1,0.45,31.2,...,1,1500,0.98,0.95,0.2,0.4,0.026,0.249,"[8.5, 10.2, 14.1, 16.8, 18.0, 19.1, 19.5, 19.4...","[12, 15, 30, 75, 180, 480, 700, 620, 390, 80, ..."
4,WB_DAR_005,Rimbick,Darjeeling,Hill Station,27.1147,88.1066,2286.0,21.0,0.84,26.5,...,1,1200,0.95,0.85,0.3,0.8,0.011,0.238,"[8.5, 10.2, 14.1, 16.8, 18.0, 19.1, 19.5, 19.4...","[12, 15, 30, 75, 180, 480, 700, 620, 390, 80, ..."


In [176]:
# ============================================
# CELL 6 — VALIDATE PREFERENCE FEATURES
# ============================================

PREFERENCE_FEATURES = [
    "nature",
    "adventure",
    "culture",
    "relaxation"
]

print("Checking Model 1 features...\n")

for feature in PREFERENCE_FEATURES:
    print(f"{feature}:")
    print(f"  Data type : {df[feature].dtype}")
    print(f"  Missing   : {df[feature].isna().sum()}")
    print(f"  Minimum   : {df[feature].min()}")
    print(f"  Maximum   : {df[feature].max()}")
    print()

Checking Model 1 features...

nature:
  Data type : float64
  Missing   : 0
  Minimum   : 0.2
  Maximum   : 1.0

adventure:
  Data type : float64
  Missing   : 0
  Minimum   : 0.2
  Maximum   : 0.95

culture:
  Data type : float64
  Missing   : 0
  Minimum   : 0.2
  Maximum   : 0.99

relaxation:
  Data type : float64
  Missing   : 0
  Minimum   : 0.4
  Maximum   : 0.95



In [177]:
# ============================================
# CELL 7 — DESTINATION VECTOR SPACE
# ============================================

destination_matrix = (
    df[PREFERENCE_FEATURES]
    .astype(float)
    .to_numpy()
)

print("Destination matrix created.")
print("Shape:", destination_matrix.shape)
print("Dimensions:", destination_matrix.shape[1])

assert destination_matrix.shape[0] == 515
assert destination_matrix.shape[1] == 4

print("\n✓ 515 destinations")
print("✓ 4 preference dimensions")
print("✓ Vector space ready")

Destination matrix created.
Shape: (515, 4)
Dimensions: 4

✓ 515 destinations
✓ 4 preference dimensions
✓ Vector space ready


In [178]:
# ============================================
# CELL 8 — INSPECT DESTINATION VECTORS
# ============================================

vector_preview = df[
    ["name", "district"] + PREFERENCE_FEATURES
].head(10)

display(vector_preview)

,name,district,nature,adventure,culture,relaxation
0,Darjeeling,Darjeeling,0.90,0.70,0.80,0.60
1,Lepchajagat,Darjeeling,0.95,0.50,0.30,0.90
2,Takdah,Darjeeling,0.95,0.60,0.40,0.90
3,Sandakphu,Darjeeling,0.98,0.95,0.20,0.40
4,Rimbick,Darjeeling,0.95,0.85,0.30,0.80
5,Kurseong,Darjeeling,0.85,0.60,0.65,0.75
6,Kalimpong,Kalimpong,0.88,0.65,0.75,0.75
7,Mirik,Darjeeling,0.85,0.55,0.40,0.85
8,Lava,Kalimpong,0.95,0.70,0.45,0.80
9,Rishyap,Kalimpong,0.97,0.60,0.30,0.90


In [179]:
# ============================================
# CELL 9 — USER PREFERENCE VECTOR
# ============================================

user_preferences = {
    "nature": 0.90,
    "adventure": 0.60,
    "culture": 0.30,
    "relaxation": 0.80
}

user_vector = np.array([
    user_preferences[feature]
    for feature in PREFERENCE_FEATURES
], dtype=float).reshape(1, -1)

print("User preferences:")
for feature, value in user_preferences.items():
    print(f"  {feature:12} = {value:.2f}")

print("\nUser vector:")
print(user_vector)

print("\nVector shape:", user_vector.shape)

assert user_vector.shape == (1, 4)
assert np.all(user_vector >= 0)
assert np.all(user_vector <= 1)

print("✓ User preference vector is valid")

User preferences:
  nature       = 0.90
  adventure    = 0.60
  culture      = 0.30
  relaxation   = 0.80

User vector:
[[0.9 0.6 0.3 0.8]]

Vector shape: (1, 4)
✓ User preference vector is valid


In [180]:
# ============================================
# CELL 10 — COSINE SIMILARITY
# ============================================

similarity_scores = cosine_similarity(
    user_vector,
    destination_matrix
)[0]

print("Similarity calculation complete.")
print("Number of scores:", len(similarity_scores))

print("\nFirst 10 similarity scores:")
print(similarity_scores[:10])

assert len(similarity_scores) == 515
assert np.all(similarity_scores >= 0)
assert np.all(similarity_scores <= 1)

print("\n✓ 515 destinations compared")
print("✓ Similarity range is valid [0, 1]")

Similarity calculation complete.
Number of scores: 515

First 10 similarity scores:
[0.9328115  0.99504832 0.99779033 0.92535757 0.9904026  0.96872759
 0.95553988 0.99540574 0.99480577 0.99905963]

✓ 515 destinations compared
✓ Similarity range is valid [0, 1]


In [181]:
# ============================================
# CELL 11 — ATTACH SIMILARITY SCORES
# ============================================

df["similarity_score"] = similarity_scores

display(
    df[
        [
            "destination_id",
            "name",
            "district",
            "nature",
            "adventure",
            "culture",
            "relaxation",
            "similarity_score"
        ]
    ].head(10)
)

,destination_id,name,district,nature,adventure,culture,relaxation,similarity_score
0,WB_DAR_001,Darjeeling,Darjeeling,0.90,0.70,0.80,0.60,0.932812
1,WB_DAR_002,Lepchajagat,Darjeeling,0.95,0.50,0.30,0.90,0.995048
2,WB_DAR_003,Takdah,Darjeeling,0.95,0.60,0.40,0.90,0.997790
3,WB_DAR_004,Sandakphu,Darjeeling,0.98,0.95,0.20,0.40,0.925358
4,WB_DAR_005,Rimbick,Darjeeling,0.95,0.85,0.30,0.80,0.990403
5,WB_DAR_006,Kurseong,Darjeeling,0.85,0.60,0.65,0.75,0.968728
6,WB_KAL_001,Kalimpong,Kalimpong,0.88,0.65,0.75,0.75,0.955540
7,WB_DAR_007,Mirik,Darjeeling,0.85,0.55,0.40,0.85,0.995406
8,WB_KAL_002,Lava,Kalimpong,0.95,0.70,0.45,0.80,0.994806
9,WB_KAL_003,Rishyap,Kalimpong,0.97,0.60,0.30,0.90,0.999060


In [182]:
# ============================================
# CELL 12 — RANK DESTINATIONS
# ============================================

ranked_destinations = (
    df
    .sort_values(
        by="similarity_score",
        ascending=False
    )
    .reset_index(drop=True)
)

ranked_destinations["rank"] = (
    ranked_destinations.index + 1
)

print("Top 10 DETOURA recommendations")
print("=" * 70)

display(
    ranked_destinations[
        [
            "rank",
            "name",
            "district",
            "category",
            "similarity_score"
        ]
    ].head(10)
)

Top 10 DETOURA recommendations


,rank,name,district,category,similarity_score
0,1,Mogra Dam Reservior,Hooghly,Nature,0.999874
1,2,Dow Hill Hot Springs,Darjeeling,Nature,0.999833
2,3,Lolegaon,Kalimpong,Hill Station,0.999742
3,4,Panchet Dam Lake Sanctuary,Paschim Bardhaman,Nature,0.999699
4,5,Tribeni Dam Reservior,Hooghly,Nature,0.999548
5,6,Rasikbil Wetlands River Banks,Cooch Behar,Nature,0.999515
6,7,Lataguri Waterfalls,Jalpaiguri,Nature,0.999499
7,8,Madhupur Hot Springs,Cooch Behar,Nature,0.999488
8,9,Mama Bhagne Rock Waterfalls,Birbhum,Nature,0.999269
9,10,Rishyap,Kalimpong,Hill Station,0.999060


In [183]:
# ============================================
# CELL 13 — MODEL 1 VALIDATION
# ============================================

print("MODEL 1 VALIDATION")
print("=" * 60)

# 1. Check number of destinations
assert len(ranked_destinations) == 515
print("✓ 515 destinations ranked")

# 2. Check similarity values
assert ranked_destinations["similarity_score"].between(0, 1).all()
print("✓ All similarity scores are within [0, 1]")

# 3. Check ranking order
scores = ranked_destinations["similarity_score"].to_numpy()

assert np.all(scores[:-1] >= scores[1:])
print("✓ Destinations correctly sorted by similarity")

# 4. Check ranks
assert ranked_destinations["rank"].tolist() == list(range(1, 516))
print("✓ Ranking numbers are correct")

# 5. Check duplicate destination IDs
assert ranked_destinations["destination_id"].is_unique
print("✓ Destination IDs are unique")

print("\nMODEL 1 VALIDATION PASSED ✓")

MODEL 1 VALIDATION
✓ 515 destinations ranked
✓ All similarity scores are within [0, 1]
✓ Destinations correctly sorted by similarity
✓ Ranking numbers are correct
✓ Destination IDs are unique

MODEL 1 VALIDATION PASSED ✓


In [184]:
# ============================================
# CELL 14 — COSINE SIMILARITY SANITY TEST
# ============================================

test_vector = np.array([
    [0.90, 0.60, 0.30, 0.80]
])

test_similarity = cosine_similarity(
    test_vector,
    test_vector
)[0][0]

print("Test vector:", test_vector[0])
print("Similarity with itself:", test_similarity)

assert np.isclose(test_similarity, 1.0)

print("\n✓ Mathematical sanity test passed")

Test vector: [0.9 0.6 0.3 0.8]
Similarity with itself: 1.0000000000000002

✓ Mathematical sanity test passed


In [185]:
# ============================================
# CELL 15 — REUSABLE RECOMMENDER
# ============================================

def recommend_destinations(
    nature,
    adventure,
    culture,
    relaxation,
    top_k=10
):
    """
    Content-based destination recommender.

    User preference vector:
    [Nature, Adventure, Culture, Relaxation]

    Returns the top-k destinations ranked
    by cosine similarity.
    """

    # Validate preference values
    preferences = [
        nature,
        adventure,
        culture,
        relaxation
    ]

    if not all(0 <= value <= 1 for value in preferences):
        raise ValueError(
            "All preference values must be between 0 and 1."
        )

    if top_k < 1:
        raise ValueError(
            "top_k must be at least 1."
        )

    # Create user vector
    user_vector = np.array(
        preferences,
        dtype=float
    ).reshape(1, -1)

    # Calculate similarity against all destinations
    similarity_scores = cosine_similarity(
        user_vector,
        destination_matrix
    )[0]

    # Copy dataset
    results = df.copy()

    # Attach similarity
    results["similarity_score"] = similarity_scores

    # Rank
    results = (
        results
        .sort_values(
            "similarity_score",
            ascending=False
        )
        .head(top_k)
        .reset_index(drop=True)
    )

    results["rank"] = (
        results.index + 1
    )

    return results

In [186]:
# ============================================
# CELL 16 — TEST RECOMMENDER FUNCTION
# ============================================

results = recommend_destinations(
    nature=0.90,
    adventure=0.60,
    culture=0.30,
    relaxation=0.80,
    top_k=10
)

display(
    results[
        [
            "rank",
            "destination_id",
            "name",
            "district",
            "category",
            "similarity_score"
        ]
    ]
)

,rank,destination_id,name,district,category,similarity_score
0,1,WB_HOG_015,Mogra Dam Reservior,Hooghly,Nature,0.999874
1,2,WB_DAR_026,Dow Hill Hot Springs,Darjeeling,Nature,0.999833
2,3,WB_KAL_004,Lolegaon,Kalimpong,Hill Station,0.999742
3,4,WB_WBD_018,Panchet Dam Lake Sanctuary,Paschim Bardhaman,Nature,0.999699
4,5,WB_HOG_006,Tribeni Dam Reservior,Hooghly,Nature,0.999548
5,6,WB_COB_011,Rasikbil Wetlands River Banks,Cooch Behar,Nature,0.999515
6,7,WB_JAL_019,Lataguri Waterfalls,Jalpaiguri,Nature,0.999499
7,8,WB_COB_013,Madhupur Hot Springs,Cooch Behar,Nature,0.999488
8,9,WB_BIR_002,Mama Bhagne Rock Waterfalls,Birbhum,Nature,0.999269
9,10,WB_KAL_003,Rishyap,Kalimpong,Hill Station,0.999060


In [187]:
# ============================================
# CELL 17 — MULTIPLE USER PROFILES
# ============================================

profiles = {
    "Nature & Relaxation": {
        "nature": 0.95,
        "adventure": 0.30,
        "culture": 0.20,
        "relaxation": 0.95
    },

    "Adventure": {
        "nature": 0.70,
        "adventure": 0.95,
        "culture": 0.20,
        "relaxation": 0.30
    },

    "Culture": {
        "nature": 0.30,
        "adventure": 0.20,
        "culture": 0.95,
        "relaxation": 0.50
    },

    "Balanced": {
        "nature": 0.70,
        "adventure": 0.70,
        "culture": 0.70,
        "relaxation": 0.70
    }
}

for profile_name, preferences in profiles.items():

    results = recommend_destinations(
        **preferences,
        top_k=5
    )

    print("\n" + "=" * 60)
    print(profile_name)
    print("=" * 60)

    display(
        results[
            [
                "rank",
                "name",
                "district",
                "similarity_score"
            ]
        ]
    )


Nature & Relaxation


,rank,name,district,similarity_score
0,1,Henry's Island,South 24 Parganas,0.993575
1,2,Lepchajagat,Darjeeling,0.987263
2,3,Hazarduari Waterfalls,Murshidabad,0.986399
3,4,Nayachar Island Coastline,Purba Medinipur,0.985496
4,5,Geonkhali Riverfront Eco Park,Purba Medinipur,0.984551



Adventure


,rank,name,district,similarity_score
0,1,Sandakphu,Darjeeling,0.986615
1,2,Pingla Patachitra Village National Park Range,Paschim Medinipur,0.959533
2,3,Kurumbera Fort Tiger Corridor,Paschim Medinipur,0.955173
3,4,Guptipara Eco Sanctuary,Hooghly,0.953677
4,5,Jhilimili Forest Jungle Camp,Bankura,0.953509



Culture


,rank,name,district,similarity_score
0,1,Deulti Cultural Hub,Howrah,0.999335
1,2,Charida Chhau Mask Village Bazaar,Purulia,0.997941
2,3,Hemtabad Ashram Complex,Uttar Dinajpur,0.995892
3,4,Baghmundi Science Center,Purulia,0.995506
4,5,Churulia Science Center,Paschim Bardhaman,0.995487



Balanced


,rank,name,district,similarity_score
0,1,Kaikhali Eco Camp Peak,South 24 Parganas,0.996232
1,2,Kalimpong,Kalimpong,0.994239
2,3,Mandarmani Cascade,Purba Medinipur,0.991579
3,4,Shankarpur Beach Cottages,Purba Medinipur,0.991231
4,5,Kurseong,Darjeeling,0.991042


# MODEL 2 — DETOURA Intelligence Engine

## Objective

Extend the content-based recommender with regional travel intelligence.

Model 2 combines:

- Content similarity
- Crowd Pressure Index (CPI)
- AI-Estimated Underrated Score (AUS)
- Season Suitability Index (SSI)

### Final Scoring Model

$$
Score_i =
w_{sim}Sim(u,v_i)
+
w_{season}SSI_{i,m}
-
w_{crowd}CPI_i
+
w_{gem}UnderratedScore_i
$$

The resulting score ranks destinations according to both traveler preference
and regional travel conditions.

### Model 2 Pipeline

Content Similarity  
+ Seasonal Suitability  
− Crowd Pressure  
+ Underratedness  
→ Final DETOURA Score

In [188]:
# ============================================
# MODEL 2 — STEP 1
# BASELINE CROWD PRESSURE INDEX
# ============================================

BASE_CPI_COLUMN = "crowd_pressure_index"

df["base_cpi"] = df[BASE_CPI_COLUMN].astype(float)

print("BASELINE CROWD PRESSURE INDEX (CPI)")
print("=" * 50)

print(f"Minimum : {df['base_cpi'].min():.6f}")
print(f"Maximum : {df['base_cpi'].max():.6f}")
print(f"Mean    : {df['base_cpi'].mean():.6f}")
print(f"Median  : {df['base_cpi'].median():.6f}")
print(f"Missing : {df['base_cpi'].isna().sum()}")

print("\n✓ Baseline CPI loaded from master dataset")
print("✓ This value represents historical/static pressure")
print("✓ Live crowd data will be incorporated separately")

BASELINE CROWD PRESSURE INDEX (CPI)
Minimum : 0.001000
Maximum : 1.000000
Mean    : 0.058643
Median  : 0.020000
Missing : 0

✓ Baseline CPI loaded from master dataset
✓ This value represents historical/static pressure
✓ Live crowd data will be incorporated separately


In [189]:
# ============================================
# BASELINE CPI VALIDATION
# ============================================

assert df["base_cpi"].notna().all(), \
    "Baseline CPI contains missing values"

assert df["base_cpi"].between(0.001, 1.0).all(), \
    "Baseline CPI values outside expected range"

print("✓ No missing baseline CPI values")
print("✓ Baseline CPI values are within [0.001, 1.0]")
print("✓ Baseline CPI validation passed")

✓ No missing baseline CPI values
✓ Baseline CPI values are within [0.001, 1.0]
✓ Baseline CPI validation passed


In [190]:
# ============================================
# MODEL 2 — STEP 1.2
# LIVE CPI ADAPTER
# ============================================

def get_live_cpi(base_cpi, live_crowd_signal=None):
    """
    Convert the baseline CPI into a dynamic CPI.

    Parameters
    ----------
    base_cpi : float
        Historical/static CPI from the master dataset.

    live_crowd_signal : float or None
        Normalized live crowd signal in [0, 1].
        None means no live signal is currently available.

    Returns
    -------
    float
        Dynamic CPI in [0.001, 1.0].
    """

    # No live signal available:
    # temporarily use the historical baseline.
    if live_crowd_signal is None:
        return float(
            np.clip(base_cpi, 0.001, 1.0)
        )

    if not 0 <= live_crowd_signal <= 1:
        raise ValueError(
            "live_crowd_signal must be between 0 and 1."
        )

    # Blend historical baseline and live signal.
    # Live signal receives greater weight.
    live_cpi = (
        0.30 * base_cpi +
        0.70 * live_crowd_signal
    )

    return float(
        np.clip(live_cpi, 0.001, 1.0)
    )

In [191]:
# ============================================
# CREATE DYNAMIC CPI
# ============================================

df["live_cpi"] = df["base_cpi"].apply(
    lambda x: get_live_cpi(x)
)

print("Dynamic CPI generated.")
print("Destinations:", len(df))

display(
    df[
        [
            "name",
            "base_cpi",
            "live_cpi"
        ]
    ].head(10)
)

Dynamic CPI generated.
Destinations: 515


,name,base_cpi,live_cpi
0,Darjeeling,0.272,0.272
1,Lepchajagat,0.025,0.025
2,Takdah,0.017,0.017
3,Sandakphu,0.026,0.026
4,Rimbick,0.011,0.011
5,Kurseong,0.070,0.070
6,Kalimpong,0.154,0.154
7,Mirik,0.054,0.054
8,Lava,0.029,0.029
9,Rishyap,0.015,0.015


In [192]:
# ============================================
# DYNAMIC CPI VALIDATION
# ============================================

assert df["live_cpi"].notna().all()

assert df["live_cpi"].between(
    0.001, 1.0
).all()

print("✓ No missing dynamic CPI values")
print("✓ Dynamic CPI remains within [0.001, 1.0]")
print("✓ Live CPI interface is ready")

✓ No missing dynamic CPI values
✓ Dynamic CPI remains within [0.001, 1.0]
✓ Live CPI interface is ready


# ============================================
# DYNAMIC CPI VALIDATION
# ============================================

assert df["live_cpi"].notna().all()

assert df["live_cpi"].between(
    0.001, 1.0
).all()

print("✓ No missing dynamic CPI values")
print("✓ Dynamic CPI remains within [0.001, 1.0]")
print("✓ Live CPI interface is ready")

In [193]:
# ============================================
# ATTRACTION DIVERSITY PROXY
# ============================================

df["attraction_diversity"] = (
    df["osm_attraction_count"]
    .astype(float)
)

print("Attraction diversity proxy created.")

display(
    df[
        [
            "name",
            "osm_attraction_count",
            "attraction_diversity"
        ]
    ].head(10)
)

Attraction diversity proxy created.


,name,osm_attraction_count,attraction_diversity
0,Darjeeling,35,35.0
1,Lepchajagat,6,6.0
2,Takdah,5,5.0
3,Sandakphu,8,8.0
4,Rimbick,4,4.0
5,Kurseong,12,12.0
6,Kalimpong,22,22.0
7,Mirik,9,9.0
8,Lava,6,6.0
9,Rishyap,4,4.0


In [194]:
# ============================================
# RAW AUS CALCULATION
# ============================================

df["AUS"] = (
    df["attraction_diversity"] *
    (1 + df["avg_ndvi"])
) / (
    1 +
    np.log(
        1 +
        df["osm_hotel_count"] +
        df["historical_review_volume_proxy"]
    )
)

print("Raw AUS calculated.")

display(
    df[
        [
            "name",
            "attraction_diversity",
            "avg_ndvi",
            "osm_hotel_count",
            "historical_review_volume_proxy",
            "AUS"
        ]
    ].head(10)
)

Raw AUS calculated.


,name,attraction_diversity,avg_ndvi,osm_hotel_count,historical_review_volume_proxy,AUS
0,Darjeeling,35.0,0.65,142,1420,6.912556
1,Lepchajagat,6.0,0.82,12,85,1.955249
2,Takdah,5.0,0.78,8,42,1.804606
3,Sandakphu,8.0,0.45,5,190,1.847688
4,Rimbick,4.0,0.84,6,25,1.648105
5,Kurseong,12.0,0.70,34,320,2.968517
6,Kalimpong,22.0,0.68,78,740,4.794966
7,Mirik,9.0,0.72,28,290,2.288184
8,Lava,6.0,0.85,16,115,1.886856
9,Rishyap,4.0,0.88,9,65,1.414202


In [195]:
# ============================================
# CALIBRATED UNDERRATED SCORE
# ============================================

df["underrated_score_calculated"] = (
    0.053894 * df["AUS"]
    + 0.149564
)

print("Calibrated underrated score calculated.")

display(
    df[
        [
            "name",
            "AUS",
            "underrated_score_calculated"
        ]
    ].head(10)
)

Calibrated underrated score calculated.


,name,AUS,underrated_score_calculated
0,Darjeeling,6.912556,0.522109
1,Lepchajagat,1.955249,0.254940
2,Takdah,1.804606,0.246821
3,Sandakphu,1.847688,0.249143
4,Rimbick,1.648105,0.238387
5,Kurseong,2.968517,0.309549
6,Kalimpong,4.794966,0.407984
7,Mirik,2.288184,0.272883
8,Lava,1.886856,0.251254
9,Rishyap,1.414202,0.225781


In [196]:
# ============================================
# AUS VALIDATION
# ============================================

assert df["AUS"].notna().all()
assert np.isfinite(df["AUS"]).all()

assert df["underrated_score_calculated"].notna().all()
assert np.isfinite(
    df["underrated_score_calculated"]
).all()

print("✓ AUS contains no NaN values")
print("✓ AUS contains no infinite values")
print("✓ Calibrated underrated scores are valid")
print("✓ AUS calculation passed")

✓ AUS contains no NaN values
✓ AUS contains no infinite values
✓ Calibrated underrated scores are valid
✓ AUS calculation passed


# MODEL 2 — STEP 3: SEASON SUITABILITY INDEX (SSI)

SSI measures destination suitability for the selected travel month
using the destination's 12-month climate profile.

### Temperature Utility

A Gaussian comfort function centered at 20°C:

$$
TempUtility =
e^{-\frac{1}{2}
\left(\frac{T-20}{5}\right)^2}
$$

### Rainfall Utility

$$
RainUtility =
\begin{cases}
1.0 & R < 50 \\
\frac{150-R}{100} & 50 \leq R \leq 150 \\
0.0 & R > 150
\end{cases}
$$

### Final SSI

$$
SSI_{i,m}=TempUtility_{i,m}\times RainUtility_{i,m}
$$

In [197]:
# ============================================
# SSI — PARSE CLIMATE ARRAYS
# ============================================

TEMP_COLUMN = "climate_temp_c"
RAIN_COLUMN = "climate_rain_mm"


def parse_monthly_array(value, column_name):
    if isinstance(value, list):
        values = value
    else:
        values = ast.literal_eval(value)

    if len(values) != 12:
        raise ValueError(
            f"{column_name} must contain exactly 12 values. "
            f"Found {len(values)}."
        )

    values = [float(x) for x in values]

    if not np.all(np.isfinite(values)):
        raise ValueError(
            f"{column_name} contains invalid numeric values."
        )

    return values


df["monthly_temperature"] = df[TEMP_COLUMN].apply(
    lambda x: parse_monthly_array(x, TEMP_COLUMN)
)

df["monthly_rainfall"] = df[RAIN_COLUMN].apply(
    lambda x: parse_monthly_array(x, RAIN_COLUMN)
)

print("✓ Temperature arrays parsed")
print("✓ Rainfall arrays parsed")
print("✓ 12 monthly values available per destination")

✓ Temperature arrays parsed
✓ Rainfall arrays parsed
✓ 12 monthly values available per destination


In [198]:
# ============================================
# SSI — TEMPERATURE UTILITY
# ============================================

COMFORT_CENTER = 20.0
TEMPERATURE_SIGMA = 5.0


def temperature_utility(temp_c):
    utility = np.exp(
        -0.5 * (
            (temp_c - COMFORT_CENTER)
            / TEMPERATURE_SIGMA
        ) ** 2
    )

    return float(
        np.clip(utility, 0.0, 1.0)
    )


print("Temperature utility sanity check:")

for temp in [5, 10, 15, 20, 25, 30, 35]:
    print(
        f"{temp:2d}°C → "
        f"{temperature_utility(temp):.4f}"
    )

Temperature utility sanity check:
 5°C → 0.0111
10°C → 0.1353
15°C → 0.6065
20°C → 1.0000
25°C → 0.6065
30°C → 0.1353
35°C → 0.0111


In [199]:
# ============================================
# SSI — RAINFALL UTILITY
# ============================================

def rainfall_utility(rainfall_mm):

    if rainfall_mm < 50:
        return 1.0

    if rainfall_mm > 150:
        return 0.0

    return float(
        (150 - rainfall_mm) / 100
    )


print("Rainfall utility sanity check:")

for rain in [0, 25, 49, 50, 75, 100, 125, 150, 200]:
    print(
        f"{rain:3d} mm → "
        f"{rainfall_utility(rain):.2f}"
    )

Rainfall utility sanity check:
  0 mm → 1.00
 25 mm → 1.00
 49 mm → 1.00
 50 mm → 1.00
 75 mm → 0.75
100 mm → 0.50
125 mm → 0.25
150 mm → 0.00
200 mm → 0.00


In [200]:
# ============================================
# SSI — CLIMATE ARRAY VALIDATION
# ============================================

assert len(df["monthly_temperature"]) == 515
assert len(df["monthly_rainfall"]) == 515

assert all(
    len(values) == 12
    for values in df["monthly_temperature"]
)

assert all(
    len(values) == 12
    for values in df["monthly_rainfall"]
)

assert all(
    np.all(np.isfinite(values))
    for values in df["monthly_temperature"]
)

assert all(
    np.all(np.isfinite(values))
    for values in df["monthly_rainfall"]
)

print("✓ 515 temperature arrays validated")
print("✓ 515 rainfall arrays validated")
print("✓ 12 months present for every destination")
print("✓ Climate dataset is ready for SSI")

✓ 515 temperature arrays validated
✓ 515 rainfall arrays validated
✓ 12 months present for every destination
✓ Climate dataset is ready for SSI


In [201]:
# ============================================
# SSI — SELECT MONTH & CALCULATE SSI
# ============================================

SELECTED_MONTH = 9
MONTH_INDEX = SELECTED_MONTH - 1

df["selected_temperature"] = df[
    "monthly_temperature"
].apply(lambda values: values[MONTH_INDEX])

df["selected_rainfall"] = df[
    "monthly_rainfall"
].apply(lambda values: values[MONTH_INDEX])

df["temperature_utility"] = df[
    "selected_temperature"
].apply(temperature_utility)

df["rainfall_utility"] = df[
    "selected_rainfall"
].apply(rainfall_utility)

df["SSI"] = (
    df["temperature_utility"] *
    df["rainfall_utility"]
)

print(f"✓ SSI calculated for month {SELECTED_MONTH}")

display(
    df[
        [
            "name",
            "selected_temperature",
            "selected_rainfall",
            "temperature_utility",
            "rainfall_utility",
            "SSI"
        ]
    ].head(10)
)

✓ SSI calculated for month 9


,name,selected_temperature,selected_rainfall,temperature_utility,rainfall_utility,SSI
0,Darjeeling,18.5,390.0,0.955997,0.0,0.0
1,Lepchajagat,18.5,390.0,0.955997,0.0,0.0
2,Takdah,18.5,390.0,0.955997,0.0,0.0
3,Sandakphu,18.5,390.0,0.955997,0.0,0.0
4,Rimbick,18.5,390.0,0.955997,0.0,0.0
5,Kurseong,18.5,390.0,0.955997,0.0,0.0
6,Kalimpong,18.5,390.0,0.955997,0.0,0.0
7,Mirik,18.5,390.0,0.955997,0.0,0.0
8,Lava,18.5,390.0,0.955997,0.0,0.0
9,Rishyap,18.5,390.0,0.955997,0.0,0.0


In [202]:
# ============================================
# SSI VALIDATION
# ============================================

assert df["SSI"].notna().all()
assert np.isfinite(df["SSI"]).all()
assert df["SSI"].between(0.0, 1.0).all()

print("✓ No missing SSI values")
print("✓ No infinite SSI values")
print("✓ SSI values are within [0, 1]")
print("✓ SSI validation passed")

✓ No missing SSI values
✓ No infinite SSI values
✓ SSI values are within [0, 1]
✓ SSI validation passed


# MODEL 2 — STEP 4: DYNAMIC MULTI-CRITERIA SCORING

The final destination score combines four intelligence signals:

1. Content-Based Similarity — preference match
2. Season Suitability Index (SSI) — climate suitability
3. Crowd Pressure Index (CPI) — infrastructure pressure
4. Underrated Score (AUS) — hidden-value potential

$$
Score_i =
w_{sim}Sim_i +
w_{season}SSI_i -
w_{crowd}CPI_i +
w_{gem}UnderratedScore_i
$$

The weights are dynamically derived from the user's travel preferences.

Higher similarity, seasonal suitability, and underrated value improve
the recommendation score.

Higher crowd pressure reduces the recommendation score.

In [203]:
# ============================================
# DYNAMIC SCORING — USER PREFERENCES
# ============================================

PREFERENCE_AXES = [
    "nature",
    "adventure",
    "culture",
    "relaxation"
]

def create_preference_vector(
    nature=0.5,
    adventure=0.5,
    culture=0.5,
    relaxation=0.5
):
    vector = np.array([
        nature,
        adventure,
        culture,
        relaxation
    ], dtype=float)

    if np.any(vector < 0) or np.any(vector > 1):
        raise ValueError(
            "Preferences must be between 0 and 1."
        )

    return vector


user_vector = create_preference_vector(
    nature=0.8,
    adventure=0.5,
    culture=0.4,
    relaxation=0.7
)

print("User preference vector:")
print(user_vector)

User preference vector:
[0.8 0.5 0.4 0.7]


In [204]:
# ============================================
# DYNAMIC WEIGHTS
# ============================================

def calculate_dynamic_weights(user_vector):

    preference_strength = np.mean(user_vector)

    w_sim = 0.40 + (0.15 * preference_strength)
    w_season = 0.20
    w_crowd = 0.20
    w_underrated = 1.0 - (
        w_sim + w_season + w_crowd
    )

    return {
        "similarity": w_sim,
        "season": w_season,
        "crowd": w_crowd,
        "underrated": w_underrated
    }


weights = calculate_dynamic_weights(
    user_vector
)

print("DYNAMIC WEIGHTS")
print("=" * 40)

for name, weight in weights.items():
    print(f"{name:15}: {weight:.4f}")

print(
    f"\nWeight total: "
    f"{sum(weights.values()):.4f}"
)

DYNAMIC WEIGHTS
similarity     : 0.4900
season         : 0.2000
crowd          : 0.2000
underrated     : 0.1100

Weight total: 1.0000


In [205]:
# ============================================
# WEIGHT VALIDATION
# ============================================

assert all(
    0 <= weight <= 1
    for weight in weights.values()
)

assert np.isclose(
    sum(weights.values()),
    1.0
)

print("✓ All weights are within [0, 1]")
print("✓ Weight sum = 1.0")
print("✓ Dynamic weighting validated")

✓ All weights are within [0, 1]
✓ Weight sum = 1.0
✓ Dynamic weighting validated


In [206]:
# ============================================
# MODEL 2 — PRE-SCORING CHECK
# ============================================

print("CURRENT DATAFRAME COLUMNS")
print("=" * 60)

for i, column in enumerate(df.columns):
    print(f"{i:3}: {column}")

CURRENT DATAFRAME COLUMNS
  0: destination_id
  1: name
  2: district
  3: category
  4: latitude
  5: longitude
  6: elevation_m
  7: slope_deg
  8: avg_ndvi
  9: accessibility_km_to_nh
 10: osm_hotel_count
 11: osm_attraction_count
 12: osm_restaurant_count
 13: historical_review_volume_proxy
 14: budget_tier
 15: estimated_daily_inr
 16: nature
 17: adventure
 18: culture
 19: relaxation
 20: crowd_pressure_index
 21: underrated_score
 22: climate_temp_c
 23: climate_rain_mm
 24: similarity_score
 25: base_cpi
 26: live_cpi
 27: attraction_diversity
 28: AUS
 29: underrated_score_calculated
 30: monthly_temperature
 31: monthly_rainfall
 32: selected_temperature
 33: selected_rainfall
 34: temperature_utility
 35: rainfall_utility
 36: SSI


In [207]:
# ============================================
# MODEL 2 — FINAL DETOURA SCORE
# ============================================

SIM_COLUMN = "similarity_score"
SSI_COLUMN = "SSI"
CPI_COLUMN = "live_cpi"
UNDERRATED_COLUMN = "underrated_score_calculated"

df["detoura_score"] = (
    weights["similarity"] * df[SIM_COLUMN]
    + weights["season"] * df[SSI_COLUMN]
    - weights["crowd"] * df[CPI_COLUMN]
    + weights["underrated"] * df[UNDERRATED_COLUMN]
)

print("✓ Final DETOURA score calculated")

display(
    df[
        [
            "name",
            SIM_COLUMN,
            SSI_COLUMN,
            CPI_COLUMN,
            UNDERRATED_COLUMN,
            "detoura_score"
        ]
    ]
    .sort_values(
        "detoura_score",
        ascending=False
    )
    .head(10)
)

✓ Final DETOURA score calculated


,name,similarity_score,SSI,live_cpi,underrated_score_calculated,detoura_score
285,Mama Bhagne Rock Green Valley,0.992293,0.0,0.260,0.822093,0.524654
460,Dow Hill Hot Springs,0.999833,0.0,0.021,0.287561,0.517350
439,Lataguri Waterfalls,0.999499,0.0,0.031,0.305135,0.517119
236,Banpara Forest Deer Park,0.996671,0.0,0.018,0.293009,0.517000
371,Raimatong Waterfalls,0.997412,0.0,0.023,0.294711,0.516550
197,Bandel Church Lake Sanctuary,0.997106,0.0,0.013,0.274288,0.516154
230,Eagle's Crag Reserve Forest,0.993606,0.0,0.017,0.296878,0.516124
368,Mongpu Waterfalls,0.998642,0.0,0.028,0.293584,0.516029
176,Jorasanko Thakurbari Green Valley,0.998262,0.0,0.018,0.275628,0.515868
390,Kalyaneswari Hill Green Valley,0.995443,0.0,0.016,0.282902,0.515686


In [208]:
# ============================================
# FINAL SCORE VALIDATION
# ============================================

assert df["detoura_score"].notna().all()
assert np.isfinite(df["detoura_score"]).all()

print("✓ No missing final scores")
print("✓ No infinite final scores")
print("✓ DETOURA scoring completed")

✓ No missing final scores
✓ No infinite final scores
✓ DETOURA scoring completed


In [209]:
# ============================================
# TOP 10 DETOURA RECOMMENDATIONS
# ============================================

top_10 = (
    df
    .sort_values(
        "detoura_score",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

top_10.index += 1

display(
    top_10[
        [
            "name",
            "district",
            "category",
            "similarity_score",
            "SSI",
            "live_cpi",
            "underrated_score_calculated",
            "detoura_score"
        ]
    ]
)

,name,district,category,similarity_score,SSI,live_cpi,underrated_score_calculated,detoura_score
1,Mama Bhagne Rock Green Valley,Birbhum,Nature,0.992293,0.0,0.260,0.822093,0.524654
2,Dow Hill Hot Springs,Darjeeling,Nature,0.999833,0.0,0.021,0.287561,0.517350
3,Lataguri Waterfalls,Jalpaiguri,Nature,0.999499,0.0,0.031,0.305135,0.517119
4,Banpara Forest Deer Park,Dakshin Dinajpur,Forest/Wildlife,0.996671,0.0,0.018,0.293009,0.517000
5,Raimatong Waterfalls,Alipurduar,Nature,0.997412,0.0,0.023,0.294711,0.516550
6,Bandel Church Lake Sanctuary,Hooghly,Nature,0.997106,0.0,0.013,0.274288,0.516154
7,Eagle's Crag Reserve Forest,Darjeeling,Forest/Wildlife,0.993606,0.0,0.017,0.296878,0.516124
8,Mongpu Waterfalls,Darjeeling,Nature,0.998642,0.0,0.028,0.293584,0.516029
9,Jorasanko Thakurbari Green Valley,Kolkata,Nature,0.998262,0.0,0.018,0.275628,0.515868
10,Kalyaneswari Hill Green Valley,Purulia,Nature,0.995443,0.0,0.016,0.282902,0.515686


In [210]:
# ============================================
# MODEL 2 — FINAL DETOURA SCORE
# ============================================

SIM_COLUMN = "similarity_score"
SSI_COLUMN = "SSI"
CPI_COLUMN = "live_cpi"
UNDERRATED_COLUMN = "underrated_score_calculated"

# Select travel month dynamically
SELECTED_MONTH = 1   # 1 = January ... 12 = December
MONTH_INDEX = SELECTED_MONTH - 1

# Calculate SSI for the selected month
df["selected_temperature"] = df[
    "monthly_temperature"
].apply(lambda values: values[MONTH_INDEX])

df["selected_rainfall"] = df[
    "monthly_rainfall"
].apply(lambda values: values[MONTH_INDEX])

df["temperature_utility"] = df[
    "selected_temperature"
].apply(temperature_utility)

df["rainfall_utility"] = df[
    "selected_rainfall"
].apply(rainfall_utility)

df["SSI"] = (
    df["temperature_utility"]
    * df["rainfall_utility"]
)

# Final DETOURA score
df["detoura_score"] = (
    weights["similarity"] * df[SIM_COLUMN]
    + weights["season"] * df[SSI_COLUMN]
    - weights["crowd"] * df[CPI_COLUMN]
    + weights["underrated"] * df[UNDERRATED_COLUMN]
)

print(f"✓ Final DETOURA score calculated")
print(f"✓ Selected travel month: {SELECTED_MONTH}")

display(
    df[
        [
            "name",
            SIM_COLUMN,
            SSI_COLUMN,
            CPI_COLUMN,
            UNDERRATED_COLUMN,
            "detoura_score"
        ]
    ]
    .sort_values(
        "detoura_score",
        ascending=False
    )
    .head(10)
)

✓ Final DETOURA score calculated
✓ Selected travel month: 1


,name,similarity_score,SSI,live_cpi,underrated_score_calculated,detoura_score
236,Banpara Forest Deer Park,0.996671,0.983931,0.018,0.293009,0.713786
270,Diamond Harbour Wildlife Range,0.995465,0.999800,0.025,0.276062,0.713105
271,Shankarpur Beach Eco Park,0.998936,0.990248,0.009,0.238827,0.711999
202,Geonkhali Riverfront Dam Reservior,0.996632,0.990248,0.015,0.243464,0.710180
85,Naihati Waterfalls,0.998621,0.983931,0.025,0.263664,0.710114
41,Kolaghat Riverfront Estuary View,0.989731,0.999200,0.017,0.248151,0.708705
428,Botanical Garden Shibpur Lake Sanctuary,0.998077,0.971611,0.029,0.279109,0.708282
317,Chandpur Beach Watchtower Forest,0.995330,0.992826,0.014,0.223852,0.708101
201,Canning Gate Waterfalls,0.995564,0.987282,0.032,0.260660,0.707555
284,Kiriteswari Jungle Camp,0.993483,0.980199,0.020,0.260406,0.707491


In [211]:
# ============================================
# SSI + FINAL SCORE DIAGNOSTIC
# ============================================

print("SSI DISTRIBUTION")
print("=" * 50)

print(f"Minimum : {df['SSI'].min():.6f}")
print(f"Maximum : {df['SSI'].max():.6f}")
print(f"Mean    : {df['SSI'].mean():.6f}")
print(f"Median  : {df['SSI'].median():.6f}")
print(f"Zero SSI: {(df['SSI'] == 0).sum()} / {len(df)}")

print("\nSEPTEMBER RAINFALL")
print("=" * 50)

print(
    df["selected_rainfall"]
    .describe()
)

print("\nFINAL SCORE")
print("=" * 50)

print(f"Minimum : {df['detoura_score'].min():.6f}")
print(f"Maximum : {df['detoura_score'].max():.6f}")
print(f"Mean    : {df['detoura_score'].mean():.6f}")

SSI DISTRIBUTION
Minimum : 0.043937
Maximum : 1.000000
Mean    : 0.795678
Median  : 0.882497
Zero SSI: 0 / 515

SEPTEMBER RAINFALL
count    515.000000
mean      11.467961
std        3.326647
min        6.000000
25%        8.000000
50%       12.000000
75%       15.000000
max       17.000000
Name: selected_rainfall, dtype: float64

FINAL SCORE
Minimum : 0.387672
Maximum : 0.713786
Mean    : 0.622577


In [212]:
# ============================================
# SSI — ALL MONTHS DIAGNOSTIC
# ============================================

monthly_ssi_summary = []

for month in range(1, 13):

    month_index = month - 1

    temp_values = df["monthly_temperature"].apply(
        lambda values: values[month_index]
    )

    rain_values = df["monthly_rainfall"].apply(
        lambda values: values[month_index]
    )

    temp_utilities = temp_values.apply(
        temperature_utility
    )

    rain_utilities = rain_values.apply(
        rainfall_utility
    )

    ssi_values = (
        temp_utilities * rain_utilities
    )

    monthly_ssi_summary.append({
        "month": month,
        "mean_temperature": temp_values.mean(),
        "mean_rainfall": rain_values.mean(),
        "mean_SSI": ssi_values.mean(),
        "max_SSI": ssi_values.max(),
        "zero_SSI_count": (ssi_values == 0).sum()
    })

ssi_monthly_df = pd.DataFrame(
    monthly_ssi_summary
)

display(ssi_monthly_df)

,month,mean_temperature,mean_rainfall,mean_SSI,max_SSI,zero_SSI_count
0,1,16.781748,11.467961,0.795678,1.000000,0
1,2,19.822136,16.130097,0.865078,1.000000,0
2,3,24.554563,24.625243,0.519061,0.915578,0
3,4,28.204466,50.178641,0.212658,0.718095,0
4,5,29.352621,120.335922,0.030967,0.087071,111
5,6,28.904272,296.190291,0.000000,0.000000,515
6,7,27.760971,395.335922,0.000000,0.000000,515
7,8,27.517282,365.302913,0.000000,0.000000,515
8,9,27.219612,288.339806,0.000000,0.000000,515
9,10,25.706408,124.155340,0.143900,0.635552,108


In [213]:
# ============================================
# FINAL SCORE DIAGNOSTICS
# ============================================

print("FINAL DETOURA SCORE DIAGNOSTICS")
print("=" * 55)

print(f"Selected month : {SELECTED_MONTH}")

print("\nSSI")
print(f"Minimum : {df['SSI'].min():.6f}")
print(f"Maximum : {df['SSI'].max():.6f}")
print(f"Mean    : {df['SSI'].mean():.6f}")
print(f"Median  : {df['SSI'].median():.6f}")
print(f"Zero SSI: {(df['SSI'] == 0).sum()} / {len(df)}")

print("\nFINAL SCORE")
print(f"Minimum : {df['detoura_score'].min():.6f}")
print(f"Maximum : {df['detoura_score'].max():.6f}")
print(f"Mean    : {df['detoura_score'].mean():.6f}")
print(f"Median  : {df['detoura_score'].median():.6f}")

FINAL DETOURA SCORE DIAGNOSTICS
Selected month : 1

SSI
Minimum : 0.043937
Maximum : 1.000000
Mean    : 0.795678
Median  : 0.882497
Zero SSI: 0 / 515

FINAL SCORE
Minimum : 0.387672
Maximum : 0.713786
Mean    : 0.622577
Median  : 0.638780


In [214]:
# ============================================
# FINAL DETOURA SCORE VALIDATION
# ============================================

assert df["detoura_score"].notna().all()
assert np.isfinite(df["detoura_score"]).all()

assert df["SSI"].between(0.0, 1.0).all()
assert df["live_cpi"].between(0.001, 1.0).all()
assert df["underrated_score_calculated"].between(0.01, 0.99).all()

print("✓ No missing final scores")
print("✓ No infinite final scores")
print("✓ SSI values valid")
print("✓ Live CPI values valid")
print("✓ Underrated scores valid")
print("✓ FINAL DETOURA SCORE VALIDATION PASSED")

✓ No missing final scores
✓ No infinite final scores
✓ SSI values valid
✓ Live CPI values valid
✓ Underrated scores valid
✓ FINAL DETOURA SCORE VALIDATION PASSED


In [215]:
# ============================================
# MODEL 2 — TOP 10 RECOMMENDATIONS
# ============================================

top_10 = (
    df
    .sort_values(
        "detoura_score",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

top_10.index = top_10.index + 1

print("TOP 10 DETOURA RECOMMENDATIONS")
print("=" * 70)
print(f"Travel month: {SELECTED_MONTH}")
print()

display(
    top_10[
        [
            "name",
            "district",
            "category",
            "similarity_score",
            "SSI",
            "live_cpi",
            "underrated_score_calculated",
            "detoura_score"
        ]
    ]
)

TOP 10 DETOURA RECOMMENDATIONS
Travel month: 1



,name,district,category,similarity_score,SSI,live_cpi,underrated_score_calculated,detoura_score
1,Banpara Forest Deer Park,Dakshin Dinajpur,Forest/Wildlife,0.996671,0.983931,0.018,0.293009,0.713786
2,Diamond Harbour Wildlife Range,South 24 Parganas,Forest/Wildlife,0.995465,0.999800,0.025,0.276062,0.713105
3,Shankarpur Beach Eco Park,Purba Medinipur,Nature,0.998936,0.990248,0.009,0.238827,0.711999
4,Geonkhali Riverfront Dam Reservior,Purba Medinipur,Nature,0.996632,0.990248,0.015,0.243464,0.710180
5,Naihati Waterfalls,North 24 Parganas,Nature,0.998621,0.983931,0.025,0.263664,0.710114
6,Kolaghat Riverfront Estuary View,Purba Medinipur,Beach,0.989731,0.999200,0.017,0.248151,0.708705
7,Botanical Garden Shibpur Lake Sanctuary,Howrah,Nature,0.998077,0.971611,0.029,0.279109,0.708282
8,Chandpur Beach Watchtower Forest,Purba Medinipur,Forest/Wildlife,0.995330,0.992826,0.014,0.223852,0.708101
9,Canning Gate Waterfalls,South 24 Parganas,Nature,0.995564,0.987282,0.032,0.260660,0.707555
10,Kiriteswari Jungle Camp,Murshidabad,Forest/Wildlife,0.993483,0.980199,0.020,0.260406,0.707491


In [216]:
# ============================================
# DETOURA — CLEAN RECOMMENDATION OUTPUT
# ============================================

recommendations = (
    df
    .sort_values(
        "detoura_score",
        ascending=False
    )
    .head(10)
    .copy()
)

recommendations["rank"] = range(
    1,
    len(recommendations) + 1
)

recommendations = recommendations[
    [
        "rank",
        "destination_id",
        "name",
        "district",
        "category",
        "latitude",
        "longitude",
        "similarity_score",
        "SSI",
        "live_cpi",
        "underrated_score_calculated",
        "detoura_score"
    ]
]

display(recommendations)

,rank,destination_id,name,district,category,latitude,longitude,similarity_score,SSI,live_cpi,underrated_score_calculated,detoura_score
236,1,WB_DDI_009,Banpara Forest Deer Park,Dakshin Dinajpur,Forest/Wildlife,25.4786,88.9072,0.996671,0.983931,0.018,0.293009,0.713786
270,2,WB_S24_014,Diamond Harbour Wildlife Range,South 24 Parganas,Forest/Wildlife,22.1155,88.3617,0.995465,0.999800,0.025,0.276062,0.713105
271,3,WB_EMD_014,Shankarpur Beach Eco Park,Purba Medinipur,Nature,22.2614,88.0936,0.998936,0.990248,0.009,0.238827,0.711999
202,4,WB_EMD_011,Geonkhali Riverfront Dam Reservior,Purba Medinipur,Nature,22.1959,87.7553,0.996632,0.990248,0.015,0.243464,0.710180
85,5,WB_N24_003,Naihati Waterfalls,North 24 Parganas,Nature,22.7980,89.0116,0.998621,0.983931,0.025,0.263664,0.710114
41,6,WB_EMD_004,Kolaghat Riverfront Estuary View,Purba Medinipur,Beach,22.1127,87.7170,0.989731,0.999200,0.017,0.248151,0.708705
428,7,WB_HWH_018,Botanical Garden Shibpur Lake Sanctuary,Howrah,Nature,22.3198,87.8197,0.998077,0.971611,0.029,0.279109,0.708282
317,8,WB_EMD_016,Chandpur Beach Watchtower Forest,Purba Medinipur,Forest/Wildlife,21.9283,87.3838,0.995330,0.992826,0.014,0.223852,0.708101
201,9,WB_S24_011,Canning Gate Waterfalls,South 24 Parganas,Nature,22.0558,88.6965,0.995564,0.987282,0.032,0.260660,0.707555
284,10,WB_MUR_013,Kiriteswari Jungle Camp,Murshidabad,Forest/Wildlife,24.3719,88.3044,0.993483,0.980199,0.020,0.260406,0.707491


In [217]:
# ============================================
# MODEL 2 — EXPORT RECOMMENDATION DATA
# ============================================

OUTPUT_COLUMNS = [
    "destination_id",
    "name",
    "district",
    "category",
    "latitude",
    "longitude",
    "similarity_score",
    "SSI",
    "live_cpi",
    "underrated_score_calculated",
    "detoura_score"
]

model2_output = df[OUTPUT_COLUMNS].copy()

model2_output = model2_output.sort_values(
    "detoura_score",
    ascending=False
).reset_index(drop=True)

model2_output.to_csv(
    "detoura_model2_recommendations.csv",
    index=False
)

print("✓ Model 2 recommendation dataset exported")
print(f"✓ Destinations: {len(model2_output)}")
print("✓ File: detoura_model2_recommendations.csv")

✓ Model 2 recommendation dataset exported
✓ Destinations: 515
✓ File: detoura_model2_recommendations.csv


In [218]:
# ============================================
# MODEL 2 — EXPORT VALIDATION
# ============================================

loaded_output = pd.read_csv(
    "detoura_model2_recommendations.csv"
)

assert len(loaded_output) == 515
assert loaded_output["destination_id"].notna().all()
assert loaded_output["detoura_score"].notna().all()

assert loaded_output["detoura_score"].is_monotonic_decreasing

print("✓ 515 destinations exported")
print("✓ Destination IDs present")
print("✓ Scores present")
print("✓ Ranking order preserved")
print("✓ MODEL 2 EXPORT VALIDATED")

✓ 515 destinations exported
✓ Destination IDs present
✓ Scores present
✓ Ranking order preserved
✓ MODEL 2 EXPORT VALIDATED
